In [1]:
import os
import csv
import logging
import shutil
import pandas as pd

# LBDB_ZWS

## Checking the source and destination folders accesibility 

In [ ]:
source_path = r"L:\NCAP"
destination_path = r"W:\BiplovTest"

# Check if the source path exists
if not os.path.exists(source_path):
    print(f"Error: The source path {source_path} does not exist.")
else:
    print(f"Source path exists: {source_path}")

# Check if the destination path exists
if not os.path.exists(destination_path):
    print(f"Error: The destination path {destination_path} does not exist.")
else:
    print(f"Destination path exists: {destination_path}")

## identifying for duplicate folders

In [ ]:
def get_sortie_names_from_folders(directory, extract_sortie=False, delimiter="#"):
    """Get a dictionary of sortie names and their respective folder paths in the given directory."""
    sorties = {}
    for folder_name in os.listdir(directory):
        folder_path = os.path.join(directory, folder_name)
        if os.path.isdir(folder_path):  # Ensure it's a folder
            if extract_sortie:  # Extract sortie for destination folder names
                sortie = folder_name.split(delimiter)[-1] if delimiter in folder_name else folder_name
            else:  # Use the full folder name for source
                sortie = folder_name
            sorties[sortie] = folder_path
    return sorties

def calculate_folder_size(folder_path):
    """Calculate the total size of a folder and its contents."""
    total_size = 0
    for dirpath, _, filenames in os.walk(folder_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_size += os.path.getsize(fp)
    return total_size

def find_duplicate_sorties_from_folders(source_path, destination_path):
    """Find sorties from source_path folders that already exist in destination_path folders."""
    source_sorties = set(get_sortie_names_from_folders(source_path, extract_sortie=False).keys())
    destination_sorties = get_sortie_names_from_folders(destination_path, extract_sortie=True)

    # Find common sorties
    duplicates = {}
    for sortie in source_sorties.intersection(destination_sorties.keys()):
        folder_path = destination_sorties[sortie]
        folder_size = calculate_folder_size(folder_path)
        duplicates[sortie] = (folder_path, folder_size)
    return duplicates

# Paths to the source and destination directories
source_path = r"L:\LBDB_ZWS\WithSortie"
destination_path = r"W:\\"

duplicates = find_duplicate_sorties_from_folders(source_path, destination_path)

if duplicates:
    print(f"Duplicate sorties found: {len(duplicates)}")
    for sortie, (folder_path, folder_size) in duplicates.items():
        print(f"Sortie: {sortie}\nDestination Path: {folder_path}\nSize: {folder_size / (1024 * 1024):.2f} MB\n")
else:
    print("No duplicate sorties found.")


## identifying and marking non-duplicate folders

In [ ]:
def add_nodupli_to_source_folders(source_path, destination_path):
    """Add '_nodupli' to source folder names that don't have duplicates in the destination."""
    source_sorties = get_sortie_names_from_folders(source_path, extract_sortie=False)
    destination_sorties = get_sortie_names_from_folders(destination_path, extract_sortie=True)

    duplicates = set(destination_sorties.keys())
    for sortie, folder_path in source_sorties.items():
        if sortie not in duplicates:
            new_folder_name = f"{sortie}_nodupli"
            new_folder_path = os.path.join(source_path, new_folder_name)
            os.rename(folder_path, new_folder_path)
            print(f"Renamed folder: {folder_path} to {new_folder_path}")

# Add '_nodupli' to source folders without duplicates
add_nodupli_to_source_folders(source_path, destination_path)

## copying the images from source to the respective destination folders after checking for duplicates

In [ ]:
def get_sortie_names_from_folders(directory, extract_sortie=False, delimiter="#"):
    """Get a dictionary of sortie names and their respective folder paths in the given directory."""
    sorties = {}
    for folder_name in os.listdir(directory):
        folder_path = os.path.join(directory, folder_name)
        if os.path.isdir(folder_path):  # Ensure it's a folder
            if extract_sortie:  # Extract sortie for destination folder names
                sortie = folder_name.split(delimiter)[-1] if delimiter in folder_name else folder_name
            else:  # Use the full folder name for source
                sortie = folder_name
            sorties[sortie] = folder_path
    return sorties

def get_new_filename(source_filename, destination_filename):
    """Rename the source file according to the destination file format."""
    # Extract the image number from the source filename (last part after '_')
    source_image_number = source_filename.split('_')[-1].split('.')[0]
    
    # Extract the image number from the destination filename (first part before '#')
    destination_image_number = destination_filename.split('#')[0]

    # Create the new filename by replacing the image number
    new_filename = destination_filename.replace(destination_image_number, source_image_number)
    return new_filename

def copy_and_rename_files(source_path, destination_path):
    """Copy and rename files from source to destination based on the SORTIE structure."""
    source_sorties = get_sortie_names_from_folders(source_path, extract_sortie=False)
    destination_sorties = get_sortie_names_from_folders(destination_path, extract_sortie=True)

    copied_files_count = 0
    skipped_files_count = 0

    for sortie, source_folder in source_sorties.items():
        if sortie in destination_sorties:
            destination_folder = destination_sorties[sortie]

            # List files in the source folder
            source_files = os.listdir(source_folder)
            # List files in the destination folder (to check for existing files)
            destination_files = os.listdir(destination_folder)

            for source_file in source_files:
                if source_file.endswith('.tif'):
                    # Full path for the source file
                    source_file_path = os.path.join(source_folder, source_file)

                    # Skip if the source file has already been copied (to prevent duplicates)
                    already_copied = False
                    for destination_file in destination_files:
                        if destination_file.endswith('.tif'):
                            # Skip files with 'modified' or 'modifiziert' in their names
                            if 'modified' in destination_file.lower() or 'modifiziert' in destination_file.lower():
                                print(f"Skipping file with 'modified' or 'modifiziert': {destination_file}")
                                skipped_files_count += 1
                                continue

                            # Get the new filename by replacing the image number
                            new_filename = get_new_filename(source_file, destination_file)
                            destination_file_path = os.path.join(destination_folder, new_filename)

                            # Check if this destination file already exists
                            if os.path.exists(destination_file_path):
                                print(f"File already exists: {destination_file_path}. Skipping.")
                                skipped_files_count += 1
                                already_copied = True
                                break

                    if already_copied:
                        continue

                    # If no duplicate found, copy and rename the file
                    try:
                        destination_file_path = os.path.join(destination_folder, new_filename)
                        shutil.copy2(source_file_path, destination_file_path)
                        print(f"Copied and renamed: {source_file_path} -> {destination_file_path}")
                        copied_files_count += 1
                    except PermissionError:
                        print(f"PermissionError: Unable to write to {destination_file_path}. Skipping.")
                        skipped_files_count += 1
                    except Exception as e:
                        print(f"Error: {e}. Skipping {source_file_path}.")
                        skipped_files_count += 1

    # Write the summary report to a CSV file
    report_path = os.path.join(destination_path, 'copy_report.csv')
    with open(report_path, mode='w', newline='') as report_file:
        csv_writer = csv.writer(report_file)
        csv_writer.writerow(['Action', 'Count'])
        csv_writer.writerow(['Copied Files', copied_files_count])
        csv_writer.writerow(['Skipped Files', skipped_files_count])
    print(f"Report generated at: {report_path}")

# Paths to the source and destination directories
source_path = r"L:\\LBDB_ZWS\\WithSortie"
destination_path = r"W:\\"

copy_and_rename_files(source_path, destination_path)


# NCAP

## identifying duplicate files

In [ ]:
import os
import csv
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

def get_sortie_names_from_folders(directory, delimiter="#"):
    """Get a dictionary of sortie names and their respective folder paths in the given directory."""
    sorties = {}
    for folder_name in os.listdir(directory):
        folder_path = os.path.join(directory, folder_name)
        if os.path.isdir(folder_path):  # Ensure it's a folder
            sortie = folder_name.split(delimiter)[-1] if delimiter in folder_name else folder_name
            sorties[sortie] = folder_path
    return sorties

def get_sortie_from_filename(filename, delimiter="_"):
    """Extract the sortie from the source filename."""
    parts = filename.split(delimiter)
    # Assuming the sortie is always the third and fourth parts (106G_3008 in this example)
    return f"{parts[2]}_{parts[3]}" if len(parts) > 3 else None

def get_new_filename(source_filename, destination_filename):
    """Generate the new filename by aligning the source file's image number with the destination format."""
    try:
        source_image_number = source_filename.split('_')[-1].split('.')[0]
        destination_image_number = destination_filename.split('#')[0]
        return destination_filename.replace(destination_image_number, source_image_number)
    except IndexError:
        logging.warning(f"Filename structure mismatch: {source_filename} or {destination_filename}")
        return None

def rename_file_with_dupli(source_file_path):
    """Rename the source file by adding '_dupli' before the file extension."""
    base_name, ext = os.path.splitext(source_file_path)
    new_name = f"{base_name}_dupli{ext}"
    os.rename(source_file_path, new_name)
    logging.info(f"Renamed duplicate file: {source_file_path} to {new_name}")
    return new_name

def identify_duplicates_by_sortie(source_path, destination_path):
    """
    Compare sorties between source filenames and destination folder names.
    If sorties match, check for duplicate files within the respective folders.
    """
    # Get sortie names from destination folders
    destination_sorties = get_sortie_names_from_folders(destination_path)

    duplicate_files = []
    unique_files = []

    # List source files
    source_files = [f for f in os.listdir(source_path) if f.endswith('.tif')]

    for source_file in source_files:
        source_file_path = os.path.join(source_path, source_file)
        is_duplicate = False

        # Extract sortie from source filename
        source_sortie = get_sortie_from_filename(source_file)
        if not source_sortie:
            logging.info(f"Skipping {source_file}: Sortie not found in filename")
            continue

        # Compare with destination folders
        for sortie, destination_folder in destination_sorties.items():
            # If sortie matches, check for duplicate files
            if source_sortie == sortie:
                destination_files = [f for f in os.listdir(destination_folder) if f.endswith('.tif')]
                
                for destination_file in destination_files:
                    new_filename = get_new_filename(source_file, destination_file)
                    if not new_filename:
                        continue

                    destination_file_path = os.path.join(destination_folder, new_filename)
                    if os.path.exists(destination_file_path):
                        logging.info(f"Duplicate file found: {source_file_path} matches {destination_file_path}")
                        duplicate_files.append((sortie, source_file, destination_folder))
                        is_duplicate = True
                        # Rename the source file to indicate it's a duplicate
                        rename_file_with_dupli(source_file_path)
                        break

                if is_duplicate:
                    break

        if not is_duplicate:
            unique_files.append((source_sortie, source_file))

    # Write the results to a CSV file
    report_path = os.path.join(destination_path, 'NCAP_duplicate_sortie_report.csv')
    with open(report_path, mode='w', newline='') as report_file:
        csv_writer = csv.writer(report_file)
        csv_writer.writerow(['Sortie', 'File Status', 'File Name', 'Destination Folder'])
        
        for sortie, file_name, destination_folder in duplicate_files:
            csv_writer.writerow([sortie, 'Duplicate', file_name, destination_folder])
        
        for sortie, file_name in unique_files:
            csv_writer.writerow([sortie, 'Unique', file_name, 'N/A'])  # 'N/A' for unique files as no destination folder is applicable

    logging.info(f"Duplicate identification complete. Report generated at: {report_path}")
    logging.info(f"Total duplicates: {len(duplicate_files)}, Total unique files: {len(unique_files)}")

# Example paths
source_path = r"L:\\NCAP"
destination_path = r"W:\\"

identify_duplicates_by_sortie(source_path, destination_path)


## copying the images from source to the respective destination folders(works)

In [2]:
# Define the Excel file path
excel_path = r"T:\\0 BIPLOV\\Bilderbestellungen_FluginfosRudi.xlsx"

# Read the Excel file
try:
    df = pd.read_excel(excel_path)
    print(df.head())  # Display the first few rows of the Excel file
except Exception as e:
    print(f"Error reading Excel file: {e}")


  SortieOriginal Bestell-ID  Bildnummer
0      23_SH_795          2        3070
1      23_SH_795          2        3071
2      23_SH_795          2        3072
3      23_SH_795          2        3073
4      23_SH_795          2        3074


In [10]:
# Function to extract date from a filename in the destination folder
def extract_date_from_filename(filename):
    parts = filename.split('#')
    if len(parts) >= 5:
        date = parts[-1].replace('.tif', '')  # Extract date and remove extension
        return date
    return None

# Function to fetch the ID from the Excel sheet based on sortie and image number
def fetch_id_from_excel(excel_path, sortie, image_number):
    df = pd.read_excel(excel_path)
    matched_row = df[(df['SortieOriginal'] == sortie) & (df['Bildnummer'] == int(image_number))]
    return matched_row['Bestell-ID'].values[0] if not matched_row.empty else None

# Function to format numbers to 4-digit format
def format_to_four_digits(number):
    return str(number).zfill(4)

# Function to copy .tif files and generate a complete report
def copy_tif_files(source_folder, destination_folder, excel_path, report_path):
    processed_files = []  # Store report details
    
    all_source_files = [f for f in os.listdir(source_folder) if f.endswith('.tif')]
    
    for filename in all_source_files:
        parts = filename.split('_')
        if len(parts) >= 4:
            sortie = '_'.join(parts[2:-1])  # Extract sortie from source filename
            image_number = format_to_four_digits(parts[-1].replace('.tif', ''))  # Extract and format image number

            flight_id = fetch_id_from_excel(excel_path, sortie, image_number)
            if flight_id is None:
                print(f"ID not found for {filename}. Skipping.")
                processed_files.append({
                    'Source Filename': filename,
                    'Transformed Filename': 'N/A',
                    'Sortie': sortie,
                    'Image Number': image_number,
                    'Flight ID': 'N/A',
                    'Date': 'N/A',
                    'Status': 'Not Found'
                })
                continue  # Skip file if no ID is found

            flight_id = "ID" + format_to_four_digits(flight_id)


            # Find the corresponding sortie in the destination folder
            file_copied = False
            for dest_folder in os.listdir(destination_folder):
                if '#' in dest_folder:
                    dest_folder_parts = dest_folder.split('#')
                    
                    # Ensure correct format before unpacking
                    if len(dest_folder_parts) != 3:
                        print(f"Skipping invalid destination folder format: {dest_folder}")
                        continue
                    
                    date, identifier, dest_sortie = dest_folder_parts
                    if sortie == dest_sortie:
                        dest_path = os.path.join(destination_folder, dest_folder)
                        for dest_file in os.listdir(dest_path):
                            if dest_file.endswith('.tif'):
                                date_str = extract_date_from_filename(dest_file)
                                if date_str:
                                    new_filename = f"{image_number}#{identifier}#{sortie}#{flight_id}#{date_str}.tif"
                                    source_file_path = os.path.join(source_folder, filename)
                                    destination_file_path = os.path.join(dest_path, new_filename)

                                    # Check if the file already exists in the destination
                                    if os.path.exists(destination_file_path):
                                        print(f"File already exists: {new_filename}. Skipping.")
                                        processed_files.append({
                                            'Source Filename': filename,
                                            'Transformed Filename': new_filename,
                                            'Sortie': sortie,
                                            'Image Number': image_number,
                                            'Flight ID': flight_id,
                                            'Date': date_str,
                                            'Status': 'Skipped (Already Exists)'
                                        })
                                        file_copied = True
                                        break
                                    else:
                                        shutil.copy(source_file_path, destination_file_path)
                                        print(f"Copied: {filename} -> {new_filename}")
                                        processed_files.append({
                                            'Source Filename': filename,
                                            'Transformed Filename': new_filename,
                                            'Sortie': sortie,
                                            'Image Number': image_number,
                                            'Flight ID': flight_id,
                                            'Date': date_str,
                                            'Status': 'Copied'
                                        })
                                        file_copied = True
                                        break
                        if file_copied:
                            break
            
            if not file_copied:
                processed_files.append({
                    'Source Filename': filename,
                    'Transformed Filename': 'N/A',
                    'Sortie': sortie,
                    'Image Number': image_number,
                    'Flight ID': flight_id,
                    'Date': 'N/A',
                    'Status': 'Not Found in Destination'
                })

        else:
            print(f"Skipping invalid filename: {filename}")
            processed_files.append({
                'Source Filename': filename,
                'Transformed Filename': 'Invalid Filename',
                'Sortie': 'N/A',
                'Image Number': 'N/A',
                'Flight ID': 'N/A',
                'Date': 'N/A',
                'Status': 'Invalid Filename'
            })

    df_report = pd.DataFrame(processed_files)
    df_report.to_csv(report_path, index=False)
    print(f"Report saved: {report_path}")

# Define paths
source_folder = r'L:\NCAP'
destination_folder = r'W:\\'
excel_path = r'T:\0 BIPLOV\Bilderbestellungen_FluginfosRudi.xlsx'
report_path = r'NCAP_processed_files_report.csv'

# Run function
copy_tif_files(source_folder, destination_folder, excel_path, report_path)


Skipping invalid destination folder format: #4035.points
Skipping invalid destination folder format: 1944_05_28#_106G_0562
Skipping invalid destination folder format: 1944_08_18#106G_2338
Skipping invalid destination folder format: 1944_08_18#_US7_3004
Skipping invalid destination folder format: 1944_09_11#_#ID8008#106G_2791
Copied: NCAP_ACIU_106G_4282_4049.tif -> 4049#00316#106G_4282#ID0412#14_02_1945.tif
Skipping invalid destination folder format: #4035.points
Skipping invalid destination folder format: 1944_05_28#_106G_0562
Skipping invalid destination folder format: 1944_08_18#106G_2338
Skipping invalid destination folder format: 1944_08_18#_US7_3004
Skipping invalid destination folder format: 1944_09_11#_#ID8008#106G_2791
Copied: NCAP_ACIU_106G_4282_4050.tif -> 4050#00316#106G_4282#ID0412#14_02_1945.tif
Skipping invalid destination folder format: #4035.points
Skipping invalid destination folder format: 1944_05_28#_106G_0562
Skipping invalid destination folder format: 1944_08_18#10